# Analyse 2: Fahrzeugmarken mit den meisten Parking Violations

Forschungsfrage: Welche Fahrzeugmarken erhalten am häufigsten Parking Violations, 
und unterscheiden sich die Top-Marken zwischen den Fiskaljahren?

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, rank
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("VehicleMakeAnalysis") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

In [6]:
processed_path = "hdfs:///parking_violations/processed/parking_violations_cleaned"

df = spark.read.parquet(processed_path)

print(f"Gesamtanzahl Zeilen: {df.count():,}")
df.printSchema()

Gesamtanzahl Zeilen: 54,220,652
root
 |-- summons_number: string (nullable = true)
 |-- plate_id: string (nullable = true)
 |-- registration_state: string (nullable = true)
 |-- issue_date: string (nullable = true)
 |-- violation_time: string (nullable = true)
 |-- violation_county: string (nullable = true)
 |-- violation_precinct: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- vehicle_make: string (nullable = true)
 |-- vehicle_body_type: string (nullable = true)
 |-- violation_code: string (nullable = true)
 |-- violation_description: string (nullable = true)
 |-- issue_date_parsed: date (nullable = true)
 |-- issue_year: integer (nullable = true)
 |-- issue_month: integer (nullable = true)
 |-- issue_weekday: integer (nullable = true)
 |-- violation_time_clean: string (nullable = true)
 |-- time_hour_raw: integer (nullable = true)
 |-- violation_minute: integer (nullable = true)
 |-- time_ampm: string (nullable = true)
 |-- violation_hour: integer (nullable

## Top 20 Fahrzeugmarken gesamt (FY2023–FY2025)

In [9]:
top_makes_overall = df \
    .filter(col("vehicle_make") != "Unknown") \
    .groupBy("vehicle_make") \
    .count() \
    .orderBy(col("count").desc()) \
    .limit(20)

top_makes_overall.show(20, truncate=False)

+------------+-------+
|vehicle_make|count  |
+------------+-------+
|HONDA       |6417854|
|TOYOT       |6374836|
|FORD        |5033828|
|NISSA       |4253147|
|CHEVR       |2897207|
|ME/BE       |2802534|
|BMW         |2678242|
|JEEP        |2485249|
|HYUND       |1866357|
|LEXUS       |1345926|
|FRUEH       |1209613|
|ACURA       |1194584|
|SUBAR       |1173960|
|KIA         |1120991|
|DODGE       |1076044|
|AUDI        |1029631|
|MAZDA       |996499 |
|VOLKS       |995872 |
|RAM         |822845 |
|INFIN       |812106 |
+------------+-------+



## Top 10 Fahrzeugmarken pro Fiskaljahr mit %-Anteil

In [10]:
from pyspark.sql.functions import sum as spark_sum, round as spark_round
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

# Gesamtanzahl pro Fiskaljahr
total_per_year = df \
    .filter(col("vehicle_make") != "Unknown") \
    .groupBy("fiscal_year") \
    .agg(spark_sum(col("vehicle_make").isNotNull().cast("int")).alias("total_year"))

# Anzahl pro Marke und Jahr
makes_per_year = df \
    .filter(col("vehicle_make") != "Unknown") \
    .groupBy("fiscal_year", "vehicle_make") \
    .count()

# Zusammenführen und %-Anteil berechnen
makes_with_pct = makes_per_year \
    .join(total_per_year, on="fiscal_year") \
    .withColumn("pct", spark_round((col("count") / col("total_year")) * 100, 2))

# Rank pro Jahr
window = Window.partitionBy("fiscal_year").orderBy(col("count").desc())
makes_ranked = makes_with_pct \
    .withColumn("rank", rank().over(window)) \
    .filter(col("rank") <= 10) \
    .orderBy("fiscal_year", "rank")

makes_ranked.select("fiscal_year", "rank", "vehicle_make", "count", "pct").show(30, truncate=False)

+-----------+----+------------+-------+-----+
|fiscal_year|rank|vehicle_make|count  |pct  |
+-----------+----+------------+-------+-----+
|2023       |1   |HONDA       |2590923|12.03|
|2023       |2   |TOYOT       |2495075|11.58|
|2023       |3   |FORD        |1981997|9.2  |
|2023       |4   |NISSA       |1769166|8.21 |
|2023       |5   |CHEVR       |1161875|5.39 |
|2023       |6   |ME/BE       |1108179|5.14 |
|2023       |7   |BMW         |1081099|5.02 |
|2023       |8   |JEEP        |997598 |4.63 |
|2023       |9   |HYUND       |736465 |3.42 |
|2023       |10  |LEXUS       |546100 |2.53 |
|2024       |1   |TOYOT       |1901022|11.82|
|2024       |2   |HONDA       |1898385|11.8 |
|2024       |3   |FORD        |1488710|9.26 |
|2024       |4   |NISSA       |1269357|7.89 |
|2024       |5   |CHEVR       |867029 |5.39 |
|2024       |6   |ME/BE       |832578 |5.18 |
|2024       |7   |BMW         |793130 |4.93 |
|2024       |8   |JEEP        |745789 |4.64 |
|2024       |9   |HYUND       |553

In [11]:
spark.stop()